# Fever and Forecast: Multimodal Dengue Early Warning for Bangladesh
## Notebook 3: Population Outbreak Forecasting & Validation Harness (Arm B, Arm C, Arm D)

This notebook trains and validates population-scale outbreak prediction models using the **1,266 district-week empirical panel** (2019–2023):

### Core Analytical Contributions:
- **Multi-Horizon Outbreak Forecasting (§M5.5):** Evaluates lead horizons at **1, 2, 4, and 8 weeks ahead** using Poisson Gradient Boosting (`count:poisson`).
- **Nested Information Hierarchy ($P_0 \to P_5$) (§M5.1):** Isolates the value of seasonal baseline ($P_0$), surveillance history ($P_1$), climate ($P_2$), Queen spatial contiguity ($P_3$), socio-economic interaction controls ($P_4$), and local clinical risk signal ($P_5$).
- **Rolling-Origin Walk-Forward Temporal CV (§M8.1):** Evaluates strictly forward-looking model stability across 2021, 2022, and the 2023 structural break year.
- **5-Block Spatial Leave-Out CV (§M8.2):** Tests geographic transferability across predefined regional holdouts (`Central`, `Eastern`, `Northern`, `Southern`, `Western`).
- **The Quantified Optimism Gap (§M8.5):** Compares standard random retrospective splits against combined temporal + spatial holdouts.
- **Arm C Multimodal Linkage (RQ3, §M7):** Tests whether local hospital clinical signals improve early warning skill within hospital catchments.
- **Arm D TreeSHAP Explainability (RQ4, §M6):** Identifies non-linear climate thresholds and interaction effects.

### Cell 1: Environment Setup, Panel Ingestion & Arm C Linkage Merge

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score
)
import xgboost as xgb

DATA_DIR = "/kaggle/working/data/processed"
panel_path = os.path.join(DATA_DIR, "master_district_weekly_panel.parquet")
signal_path = os.path.join(DATA_DIR, "clinical_risk_signal_weekly.parquet")

print(f"Reading master empirical panel from: {panel_path}")
df_panel = pd.read_parquet(panel_path)

print(f"Reading weekly clinical risk signal from: {signal_path}")
df_signal = pd.read_parquet(signal_path)

merge_cols = ["district", "year", "epi_week"]
df_merged = pd.merge(
    df_panel,
    df_signal[["district", "year", "epi_week", "clinical_risk_score_mean", "clinical_test_volume"]],
    on=merge_cols,
    how="left"
)

df_merged["clinical_risk_score_mean"] = df_merged["clinical_risk_score_mean"].fillna(0.0)
df_merged["clinical_test_volume"] = df_merged["clinical_test_volume"].fillna(0.0)

print("\n" + "=" * 70)
print("✅ Cell 1 Complete! Master Outbreak Panel & Clinical Signal Ready:")
print(f"• Total Master Panel Records: {len(df_merged)} district-weeks")
print(f"• Total Features Available  : {df_merged.shape[1]} columns")
print(f"• Surveillance Years        : {sorted(df_merged['year'].unique().tolist())}")
print(f"• Districts in Panel        : {df_merged['district'].unique().tolist()}")
print(f"• Spatial Blocks Covered    : {df_merged['spatial_block'].unique().tolist()}")
print(f"• Arm C Catchments Linked   : {df_signal['district'].unique().tolist()} ({len(df_signal)} weekly records)")
print("=" * 70)

### Cell 2: Feature Ladder Configuration ($P_0 \to P_5$)

In [ ]:
# 1. Cyclical seasonal coordinates
df_merged["sin_week"] = np.sin(2 * np.pi * df_merged["epi_week"] / 52.0)
df_merged["cos_week"] = np.cos(2 * np.pi * df_merged["epi_week"] / 52.0)

# Non-linear climate interactions
if "temp_mean" in df_merged.columns and "rainfall_total" in df_merged.columns:
    df_merged["temp_x_rain"] = df_merged["temp_mean"] * df_merged["rainfall_total"]
else:
    df_merged["temp_x_rain"] = 0.0

if "temp_mean" in df_merged.columns and "humidity_mean" in df_merged.columns:
    df_merged["temp_x_humidity"] = df_merged["temp_mean"] * df_merged["humidity_mean"]
else:
    df_merged["temp_x_humidity"] = 0.0

# 2. Define Nested Information Tiers (§M5.1)
# P0: Seasonal Baseline (the mandatory floor)
p0_feats = ["sin_week", "cos_week", "epi_week"]

# P1: + Autoregressive Surveillance Lags (1-8 weeks)
case_lags = [c for c in df_merged.columns if any(k in c for k in ["cases_lag_", "incidence_lag_"])]
p1_feats = p0_feats + sorted(case_lags)

# P2: + Climate Covariates, Lags (1-4 weeks), and Cumulative Rainfall
climate_base = [c for c in ["temp_mean", "temp_min", "temp_max", "rainfall_total", "humidity_mean", "rainfall_accum_2w", "rainfall_accum_3w"] if c in df_merged.columns]
climate_lags = [c for c in df_merged.columns if any(k in c for k in ["temp_mean_lag", "temp_min_lag", "temp_max_lag", "rainfall_total_lag", "humidity_mean_lag"])]
p2_feats = p1_feats + sorted(list(set(climate_base + climate_lags)))

# P3: + Queen Spatial Contiguity Neighbor Lags (1-4 weeks)
spatial_lags = [c for c in df_merged.columns if "spatial_lag_cases_" in c]
p3_feats = p2_feats + sorted(spatial_lags)

# P4: + Socio-Economic Controls & Non-Linear Climate Interactions
socio_feats = [c for c in ["poverty_headcount_pct", "urbanization_rate_pct", "hospital_beds_per_10k", "temp_x_rain", "temp_x_humidity"] if c in df_merged.columns]
p4_feats = p3_feats + sorted(socio_feats)

# P5: + Local Clinical Diagnostic Signal S_hat_{d,t} (Arm C Linkage / RQ3)
clinical_feats = ["clinical_risk_score_mean", "clinical_test_volume"]
p5_feats = p4_feats + clinical_feats

population_tiers = {
    "P0_Seasonal_Floor": p0_feats,
    "P1_Surveillance_Lags": p1_feats,
    "P2_Plus_Climate": p2_feats,
    "P3_Plus_Spatial_Queen": p3_feats,
    "P4_Plus_Socio_Interactions": p4_feats,
    "P5_Multimodal_ArmC_Linkage": p5_feats
}

print("=" * 70)
print("✅ Cell 2 Complete! Population Nested Hierarchy (P0 -> P5) Configured:")
print("=" * 70)
for tier_name, feats in population_tiers.items():
    print(f"• {tier_name:<28} : {len(feats):>2} features")
print("=" * 70)
print(f"Target horizons configured: 1, 2, 4, and 8 weeks ahead (§M5.5)")

### Cell 3: Multi-Horizon Forecasting Hierarchy ($P_0 \to P_5$)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score
)

HORIZONS = [1, 2, 4, 8]
results = []

print("=" * 80)
print("TRAINING MULTI-HORIZON POPULATION FORECASTING HIERARCHY (P0 -> P5)")
print("Condition 1: Single Train/Test Split (Comparable to Shiddik et al., 2026)")
print("=" * 80)

for h in HORIZONS:
    target_count_col = f"target_cases_lead_{h}"
    target_outbreak_col = f"target_outbreak_relative_lead_{h}"
    valid_mask = df_merged[target_count_col].notna() & df_merged[target_outbreak_col].notna()
    sub_df = df_merged[valid_mask].copy()
    
    train_df, test_df = train_test_split(sub_df, test_size=0.20, random_state=42)
    p0_rmse = None
    
    for tier_name, feats in population_tiers.items():
        X_train = train_df[feats].fillna(0).values
        y_train_count = train_df[target_count_col].values
        y_train_outbreak = train_df[target_outbreak_col].values
        
        X_test = test_df[feats].fillna(0).values
        y_test_count = test_df[target_count_col].values
        y_test_outbreak = test_df[target_outbreak_col].values
        
        model_count = xgb.XGBRegressor(objective="count:poisson", n_estimators=120, max_depth=4, learning_rate=0.08, random_state=42)
        model_count.fit(X_train, y_train_count)
        pred_counts = np.clip(model_count.predict(X_test), 0, None)
        
        rmse = float(np.sqrt(mean_squared_error(y_test_count, pred_counts)))
        mae = float(mean_absolute_error(y_test_count, pred_counts))
        r2 = float(r2_score(y_test_count, pred_counts))
        
        if tier_name == "P0_Seasonal_Floor":
            p0_rmse = rmse
            skill_score = 0.0
        else:
            skill_score = float(1.0 - (rmse / p0_rmse)) if p0_rmse and p0_rmse > 0 else 0.0
            
        model_outbreak = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
        model_outbreak.fit(X_train, y_train_outbreak)
        pred_probs = model_outbreak.predict_proba(X_test)[:, 1]
        pred_binary = (pred_probs >= 0.5).astype(int)
        
        auc = float(roc_auc_score(y_test_outbreak, pred_probs)) if len(np.unique(y_test_outbreak)) > 1 else np.nan
        pr_auc = float(average_precision_score(y_test_outbreak, pred_probs)) if len(np.unique(y_test_outbreak)) > 1 else np.nan
        acc = float(accuracy_score(y_test_outbreak, pred_binary))
        brier = float(brier_score_loss(y_test_outbreak, pred_probs))
        
        results.append({
            "horizon_weeks": h,
            "tier": tier_name,
            "n_features": len(feats),
            "rmse": round(rmse, 2),
            "mae": round(mae, 2),
            "r2": round(r2, 4),
            "skill_vs_p0": round(skill_score, 4),
            "outbreak_auc": round(auc, 4),
            "outbreak_pr_auc": round(pr_auc, 4),
            "outbreak_accuracy": round(acc, 4),
            "brier_score": round(brier, 4)
        })
        
    print(f"✅ Lead Horizon h={h} week(s) evaluated across P0 -> P5.")

df_hierarchy = pd.DataFrame(results)
hierarchy_path = os.path.join(DATA_DIR, "population_model_hierarchy_metrics.csv")
df_hierarchy.to_csv(hierarchy_path, index=False)

print("\n" + "=" * 80)
print(f"✅ Cell 3 Complete! Saved hierarchy metrics -> {hierarchy_path}")
print("=" * 80)
cols_show = ["tier", "rmse", "mae", "r2", "skill_vs_p0", "outbreak_auc", "outbreak_pr_auc"]
print(df_hierarchy[df_hierarchy["horizon_weeks"] == 4][cols_show].to_string(index=False))

### Cell 4: Rolling-Origin Walk-Forward Temporal Validation (2020–2023) (§M8.1)

In [ ]:
TEST_YEARS = [2021, 2022, 2023]
TARGET_H = 4

target_count_col = f"target_cases_lead_{TARGET_H}"
target_outbreak_col = f"target_outbreak_relative_lead_{TARGET_H}"
valid_df = df_merged[df_merged[target_count_col].notna() & df_merged[target_outbreak_col].notna()].copy()

rolling_records = []
print("=" * 80)
print(f"EVALUATING ROLLING-ORIGIN WALK-FORWARD TEMPORAL CV (h = {TARGET_H} Weeks Ahead)")
print("Condition 2: Strictly Causal Time-Series Cross-Validation (§M8.1)")
print("=" * 80)

for test_yr in TEST_YEARS:
    train_split = valid_df[valid_df["year"] < test_yr]
    test_split = valid_df[valid_df["year"] == test_yr]
    if len(train_split) == 0 or len(test_split) == 0:
        continue
    p0_rmse = None
    
    for tier_name in ["P0_Seasonal_Floor", "P1_Surveillance_Lags", "P2_Plus_Climate", "P4_Plus_Socio_Interactions", "P5_Multimodal_ArmC_Linkage"]:
        feats = population_tiers[tier_name]
        X_tr = train_split[feats].fillna(0).values
        y_tr_cnt = train_split[target_count_col].values
        y_tr_out = train_split[target_outbreak_col].values
        
        X_te = test_split[feats].fillna(0).values
        y_te_cnt = test_split[target_count_col].values
        y_te_out = test_split[target_outbreak_col].values
        
        cnt_model = xgb.XGBRegressor(objective="count:poisson", n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
        cnt_model.fit(X_tr, y_tr_cnt)
        preds_cnt = np.clip(cnt_model.predict(X_te), 0, None)
        
        rmse = float(np.sqrt(mean_squared_error(y_te_cnt, preds_cnt)))
        mae = float(mean_absolute_error(y_te_cnt, preds_cnt))
        r2 = float(r2_score(y_te_cnt, preds_cnt)) if len(y_te_cnt) > 1 else 0.0
        
        if tier_name == "P0_Seasonal_Floor":
            p0_rmse = rmse
            skill = 0.0
        else:
            skill = float(1.0 - (rmse / p0_rmse)) if p0_rmse and p0_rmse > 0 else 0.0
            
        out_model = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
        out_model.fit(X_tr, y_tr_out)
        preds_prob = out_model.predict_proba(X_te)[:, 1]
        
        auc = float(roc_auc_score(y_te_out, preds_prob)) if len(np.unique(y_te_out)) > 1 else np.nan
        pr_auc = float(average_precision_score(y_te_out, preds_prob)) if len(np.unique(y_te_out)) > 1 else np.nan
        brier = float(brier_score_loss(y_te_out, preds_prob))
        
        rolling_records.append({
            "test_year": test_yr,
            "train_period": f"2019-{test_yr-1}",
            "tier": tier_name,
            "rmse": round(rmse, 2),
            "mae": round(mae, 2),
            "r2": round(r2, 4),
            "skill_vs_p0": round(skill, 4),
            "outbreak_auc": round(auc, 4),
            "outbreak_pr_auc": round(pr_auc, 4),
            "brier_score": round(brier, 4)
        })
        
    print(f"✅ Fold complete: Trained on 2019–{test_yr-1} -> Evaluated on unseen {test_yr} (n={len(test_split)} district-weeks)")

df_rolling = pd.DataFrame(rolling_records)
rolling_path = os.path.join(DATA_DIR, "population_rolling_origin_metrics.csv")
df_rolling.to_csv(rolling_path, index=False)

print("\n" + "=" * 80)
print(f"✅ Cell 4 Complete! Saved rolling-origin metrics -> {rolling_path}")
print("=" * 80)
cols_view = ["test_year", "tier", "rmse", "mae", "r2", "outbreak_auc", "outbreak_pr_auc"]
print(df_rolling[cols_view].to_string(index=False))

### Cell 5: 5-Block Spatial Leave-Out CV & The Optimism Gap Matrix (§M8.2, §M8.5)

In [ ]:
TARGET_H = 4
target_count_col = f"target_cases_lead_{TARGET_H}"
target_outbreak_col = f"target_outbreak_relative_lead_{TARGET_H}"
valid_df = df_merged[df_merged[target_count_col].notna() & df_merged[target_outbreak_col].notna()].copy()
spatial_blocks = [b for b in valid_df["spatial_block"].unique() if pd.notna(b)]
model_feats = population_tiers["P4_Plus_Socio_Interactions"]

spatial_records = []
all_spatial_y_true_out, all_spatial_y_pred_out = [], []
all_spatial_y_true_cnt, all_spatial_y_pred_cnt = [], []

for block in spatial_blocks:
    tr_df = valid_df[valid_df["spatial_block"] != block]
    te_df = valid_df[valid_df["spatial_block"] == block]
    if len(te_df) == 0:
        continue
        
    X_tr = tr_df[model_feats].fillna(0).values
    y_tr_cnt = tr_df[target_count_col].values
    y_tr_out = tr_df[target_outbreak_col].values
    
    X_te = te_df[model_feats].fillna(0).values
    y_te_cnt = te_df[target_count_col].values
    y_te_out = te_df[target_outbreak_col].values
    
    cnt_mod = xgb.XGBRegressor(objective="count:poisson", n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
    cnt_mod.fit(X_tr, y_tr_cnt)
    p_cnt = np.clip(cnt_mod.predict(X_te), 0, None)
    
    out_mod = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
    out_mod.fit(X_tr, y_tr_out)
    p_out = out_mod.predict_proba(X_te)[:, 1]
    
    all_spatial_y_true_cnt.extend(y_te_cnt)
    all_spatial_y_pred_cnt.extend(p_cnt)
    all_spatial_y_true_out.extend(y_te_out)
    all_spatial_y_pred_out.extend(p_out)
    
    auc = float(roc_auc_score(y_te_out, p_out)) if len(np.unique(y_te_out)) > 1 else np.nan
    rmse = float(np.sqrt(mean_squared_error(y_te_cnt, p_cnt)))
    
    spatial_records.append({
        "held_out_spatial_block": block,
        "n_test_district_weeks": len(te_df),
        "test_districts": te_df["district"].unique().tolist(),
        "outbreak_auc": round(auc, 4) if not np.isnan(auc) else None,
        "rmse": round(rmse, 2)
    })

df_spatial = pd.DataFrame(spatial_records)
spatial_path = os.path.join(DATA_DIR, "population_spatial_holdout_metrics.csv")
df_spatial.to_csv(spatial_path, index=False)

cond3_auc = float(roc_auc_score(all_spatial_y_true_out, all_spatial_y_pred_out))
cond3_rmse = float(np.sqrt(mean_squared_error(all_spatial_y_true_cnt, all_spatial_y_pred_cnt)))

# Condition 4: Combined Temporal + Spatial Holdout
cond4_y_true_out, cond4_y_pred_out = [], []
cond4_y_true_cnt, cond4_y_pred_cnt = [], []

for block in spatial_blocks:
    tr_df = valid_df[(valid_df["spatial_block"] != block) & (valid_df["year"] < 2023)]
    te_df = valid_df[(valid_df["spatial_block"] == block) & (valid_df["year"] == 2023)]
    if len(tr_df) == 0 or len(te_df) == 0:
        continue
        
    X_tr = tr_df[model_feats].fillna(0).values
    y_tr_cnt = tr_df[target_count_col].values
    y_tr_out = tr_df[target_outbreak_col].values
    
    X_te = te_df[model_feats].fillna(0).values
    y_te_cnt = te_df[target_count_col].values
    y_te_out = te_df[target_outbreak_col].values
    
    cnt_mod = xgb.XGBRegressor(objective="count:poisson", n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
    cnt_mod.fit(X_tr, y_tr_cnt)
    p_cnt = np.clip(cnt_mod.predict(X_te), 0, None)
    
    out_mod = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
    out_mod.fit(X_tr, y_tr_out)
    p_out = out_mod.predict_proba(X_te)[:, 1]
    
    cond4_y_true_cnt.extend(y_te_cnt)
    cond4_y_pred_cnt.extend(p_cnt)
    cond4_y_true_out.extend(y_te_out)
    cond4_y_pred_out.extend(p_out)

cond4_auc = float(roc_auc_score(cond4_y_true_out, cond4_y_pred_out))
cond4_rmse = float(np.sqrt(mean_squared_error(cond4_y_true_cnt, cond4_y_pred_cnt)))

cond1_auc = float(df_hierarchy[(df_hierarchy["horizon_weeks"] == TARGET_H) & (df_hierarchy["tier"] == "P4_Plus_Socio_Interactions")]["outbreak_auc"].iloc[0])
cond1_rmse = float(df_hierarchy[(df_hierarchy["horizon_weeks"] == TARGET_H) & (df_hierarchy["tier"] == "P4_Plus_Socio_Interactions")]["rmse"].iloc[0])
cond2_auc = float(df_rolling[(df_rolling["tier"] == "P4_Plus_Socio_Interactions")]["outbreak_auc"].dropna().mean())
cond2_rmse = float(df_rolling[(df_rolling["tier"] == "P4_Plus_Socio_Interactions")]["rmse"].mean())

optimism_gap_auc = cond1_auc - cond4_auc
optimism_gap_rmse = cond4_rmse - cond1_rmse

df_matrix = pd.DataFrame([
    {
        "Evaluation Dimension": "National (All Districts)",
        "Single Random Split (Shiddik-Comparable)": f"AUC: {cond1_auc:.4f} | RMSE: {cond1_rmse:.1f}",
        "Rolling-Origin Temporal Holdout": f"AUC: {cond2_auc:.4f} | RMSE: {cond2_rmse:.1f}"
    },
    {
        "Evaluation Dimension": "Spatial Leave-Out (5 Blocks)",
        "Single Random Split (Shiddik-Comparable)": f"AUC: {cond3_auc:.4f} | RMSE: {cond3_rmse:.1f}",
        "Rolling-Origin Temporal Holdout": f"AUC: {cond4_auc:.4f} | RMSE: {cond4_rmse:.1f} [FULLY HONEST]"
    }
])
matrix_path = os.path.join(DATA_DIR, "population_optimism_gap_matrix.csv")
df_matrix.to_csv(matrix_path, index=False)

print("\n" + "=" * 80)
print("🎯 TABLE M8.5: COMBINED POPULATION VALIDATION MATRIX & THE OPTIMISM GAP")
print("=" * 80)
print(df_matrix.to_string(index=False))
print("\n" + "-" * 80)
print(f"🔥 THE HEADLINE RESULT (§4.6): Optimism Gap = Δ AUC: +{optimism_gap_auc:.4f} | Δ RMSE: +{optimism_gap_rmse:.1f}")
print("-" * 80)

### Cell 6: Arm C Multimodal Linkage (RQ3) & TreeSHAP Attribution (Arm D)

In [ ]:
import shap

TARGET_H = 4
target_count_col = f"target_cases_lead_{TARGET_H}"
target_outbreak_col = f"target_outbreak_relative_lead_{TARGET_H}"
valid_df = df_merged[df_merged[target_count_col].notna() & df_merged[target_outbreak_col].notna()].copy()

# 1. Arm C Linkage
catchment_districts = ["Dhaka", "Mymensingh"]
df_catchment = valid_df[valid_df["district"].isin(catchment_districts)].copy()
tr_catch = df_catchment[df_catchment["year"] < 2023]
te_catch = df_catchment[df_catchment["year"] == 2023]

m_no_clin = xgb.XGBRegressor(objective="count:poisson", n_estimators=120, max_depth=4, learning_rate=0.08, random_state=42)
m_no_clin.fit(tr_catch[population_tiers["P4_Plus_Socio_Interactions"]].fillna(0).values, tr_catch[target_count_col].values)
preds_no_clin = np.clip(m_no_clin.predict(te_catch[population_tiers["P4_Plus_Socio_Interactions"]].fillna(0).values), 0, None)

m_with_clin = xgb.XGBRegressor(objective="count:poisson", n_estimators=120, max_depth=4, learning_rate=0.08, random_state=42)
m_with_clin.fit(tr_catch[population_tiers["P5_Multimodal_ArmC_Linkage"]].fillna(0).values, tr_catch[target_count_col].values)
preds_with_clin = np.clip(m_with_clin.predict(te_catch[population_tiers["P5_Multimodal_ArmC_Linkage"]].fillna(0).values), 0, None)

y_actual = te_catch[target_count_col].values
rmse_no = np.sqrt(mean_squared_error(y_actual, preds_no_clin))
mae_no = mean_absolute_error(y_actual, preds_no_clin)
rmse_with = np.sqrt(mean_squared_error(y_actual, preds_with_clin))
mae_with = mean_absolute_error(y_actual, preds_with_clin)

df_linkage = pd.DataFrame([{
    "Catchment": "Dhaka & Jamalpur/Mymensingh",
    "Forecast Horizon": f"{TARGET_H} Weeks Ahead",
    "Test Year": 2023,
    "P4 RMSE": round(rmse_no, 1),
    "P5 RMSE": round(rmse_with, 1),
    "P4 MAE": round(mae_no, 1),
    "P5 MAE": round(mae_with, 1)
}])
linkage_path = os.path.join(DATA_DIR, "arm_c_clinical_linkage_evaluation.csv")
df_linkage.to_csv(linkage_path, index=False)
print(df_linkage.to_string(index=False))

# 2. Arm D TreeSHAP
explainer = shap.TreeExplainer(cnt_mod)
shap_values = explainer.shap_values(valid_df[p4_feats].fillna(0).values[:300])
shap_df = pd.DataFrame({
    "feature": p4_feats,
    "mean_abs_shap_attribution": np.mean(np.abs(shap_values), axis=0)
}).sort_values("mean_abs_shap_attribution", ascending=False).reset_index(drop=True)

shap_path = os.path.join(DATA_DIR, "shap_climate_feature_importance.csv")
shap_df.to_csv(shap_path, index=False)

print("\n" + "=" * 80)
print("🎯 NOTEBOOK 3 COMPLETE: ALL POPULATION FORECASTING DELIVERABLES VERIFIED")
print("=" * 80)
expected_files = [
    "population_model_hierarchy_metrics.csv",
    "population_rolling_origin_metrics.csv",
    "population_spatial_holdout_metrics.csv",
    "population_optimism_gap_matrix.csv",
    "arm_c_clinical_linkage_evaluation.csv",
    "shap_climate_feature_importance.csv"
]
for f in expected_files:
    f_p = os.path.join(DATA_DIR, f)
    if os.path.exists(f_p):
        kb = os.path.getsize(f_p) / 1024
        print(f"  ✅ [READY] {f:<42} : {kb:.1f} KB")
    else:
        print(f"  ❌ [MISSING] {f}")